# Lab 2: sequential overnight launcher

This is the revised implementation, not the earlier baseline. Run it in a fresh GPU kernel,
one notebook at a time. A nominal 16-GB GPU is the lab target. The RTX 4050 uses a CPU-offloaded
low-VRAM profile, which must pass the real preflight on your own PC. A cold setup needs internet,
roughly 40 GB free disk overall and enough host RAM for offload. No hosted model API is used.

The default is a full run. `LAB2_SMOKE_ONLY=1` explicitly skips model-backed measurements;
it is a developer check, never a completed experiment. Real human ratings are never generated.
Device-memory sampling covers the actual subprocess stages but is not proof that an RTX 4090
is a 16-GB GPU. Keep the printed hardware identity and all outputs.

Before leaving the computer, run through the real GPU preflight and confirm it passes.
Matching completed items resume from atomic checkpoints. Changed code, data or hardware starts
a separate checkpoint namespace; no old result is relabeled. Keep the machine awake and the
local GPU runtime running. Closing the local server or rebooting stops execution.

This runs the revised Part 1 and Part 3 notebooks in order. It does not repeat your completed Part 2 training. Executed notebooks autosave after every cell, and the result ZIPs include the finished copies. Real ratings and external submission steps are not automated.

In [1]:
from pathlib import Path
import os, subprocess, sys, platform, json, time

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SMOKE_ONLY = os.environ.get("LAB2_SMOKE_ONLY", "0") == "1"
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / "tools/setup_runtime.py").exists()), None)
if ROOT is None:
    if os.environ.get("LAB2_ROOT"):
        ROOT = Path(os.environ["LAB2_ROOT"]).expanduser().resolve()
    elif Path("/content").exists():
        ROOT = Path("/content/Lab-2")
    elif Path("/kaggle/working").exists():
        ROOT = Path("/kaggle/working/Lab-2")
    else:
        ROOT = Path.home() / "DATA266" / "Lab-2"
    if not ROOT.exists():
        ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").exists():
        raise RuntimeError(f"{ROOT} is an older/incomplete checkout. Update it with git pull, then restart this notebook.")
print("Notebook host:", platform.python_version(), "repository:", ROOT)
print("Mode:", "SMOKE ONLY: no full training or model-backed evaluation" if SMOKE_ONLY else "FULL RUN")
# Refresh a clean main checkout, but never overwrite local edits or reset a branch.
if not SMOKE_ONLY and os.environ.get("LAB2_UPDATE_REPOSITORY", "1") == "1":
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
    if branch == "main" and not dirty:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    else:
        print("Kept existing checkout: local edits or a non-main branch. No files were reset.")
if "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
    # A freshly downloaded notebook must not silently execute the old broken installer.
    previous_root = ROOT
    ROOT = ROOT.with_name(ROOT.name + "-notebook-v2")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/denishatank12/Lab-2.git", str(ROOT)], check=True)
    if not (ROOT / "tools/setup_runtime.py").is_file() or "SETUP_SCHEMA = 2" not in (ROOT / "tools/setup_runtime.py").read_text(encoding="utf-8"):
        raise RuntimeError(f"Update the checkout at {ROOT} before running. The original checkout was preserved.")
    print("Using repaired code at", ROOT, "; the older checkout and its results remain at", previous_root)
subprocess.run([sys.executable, str(ROOT / "tools/setup_runtime.py"),
                "--device", "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")], check=True)
requested_device = "cpu" if SMOKE_ONLY else os.environ.get("LAB2_DEVICE", "cuda")
import shutil
use_cuda = requested_device == "cuda" or (requested_device == "auto" and shutil.which("nvidia-smi") is not None)
runtime_file = ROOT / (".lab2_runtime_cuda.json" if use_cuda else ".lab2_runtime_cpu.json")
PYTHON = Path(json.loads(runtime_file.read_text())["python"])
SEED = 266
print("All remaining commands use:", PYTHON)

def run_script(relative, *arguments):
    command = [str(PYTHON), "-u", str(ROOT / relative), *map(str, arguments)]
    logs = ROOT / "runtime_checks"
    logs.mkdir(exist_ok=True)
    log = logs / (Path(relative).stem + ".log")
    print("Running", Path(relative).name, flush=True)
    with log.open("a", encoding="utf-8") as handle:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in process.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f"{Path(relative).name} stopped (exit {code}). Full diagnostic log: {log}")

# Permit first-time downloads during preflight, even after restarting this notebook.
if not SMOKE_ONLY:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
# Monitor actual device memory in each subprocess. A larger GPU is not relabeled as a T4.
_run_stage = run_script
def run_script(relative, *arguments):
    record = ROOT / "runtime_checks" / (Path(relative).stem + "_" + str(time.time_ns()) + ".memory.json")
    _run_stage("tools/monitored_run.py", "--record", record, "--", PYTHON, ROOT / relative, *arguments)


Notebook host: 3.12.6 repository: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2
Mode: SMOKE ONLY: no full training or model-backed evaluation


{
  "commit": "a0d7dda",
  "working_tree_dirty": true,
  "python": "3.12.6",
  "seed": 266,
  "executable": "/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python",
  "versions": {
    "torch": "2.5.1",
    "torchvision": "0.20.1",
    "diffusers": "0.31.0",
    "transformers": "4.46.2",
    "peft": "0.13.2",
    "datasets": "3.1.0",
    "huggingface-hub": "0.26.2"
  },
  "cuda_available": false,
  "gpu": "cpu"
}
PASS: core imports, matching torchvision kernels and device preflight


LAB2_PYTHON=/Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python
All remaining commands use: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/.venv/bin/python


## Before walking away
Run this cell while you are still at the PC. Enter the Hugging Face read token securely if needed. It downloads the selected original adapter and runs both real GPU preflights. Do not leave until this cell prints that both preflights passed. For this RTX 4050, keep the PC on power with sleep disabled and the local runtime/server running.

In [2]:
DATA = Path(os.environ.get("LAB2_DATA_DIR", str(ROOT / "part1_rag/data"))).resolve()
if SMOKE_ONLY:
    print("SKIPPED: all real GPU preflights. This is not an overnight-ready GPU result.")
else:
    from getpass import getpass
    adapter = ROOT / "part3_agents/outputs/hub_adapter"
    if not os.environ.get("HF_TOKEN") and not (adapter / "hub_provenance.json").is_file():
        os.environ["HF_TOKEN"] = getpass("Hugging Face read token for your private Part 2 adapter (hidden): ")
    run_script("part3_agents/src/prepare_evaluation.py", "--download-adapter")
    run_script("tools/overnight_preflight.py", "--part", "part1_rag", "--data-dir", DATA)
    run_script("tools/overnight_preflight.py", "--part", "part3_agents")
    print("BOTH REAL GPU PREFLIGHTS PASSED. You can now start the next cell.")

SKIPPED: all real GPU preflights. This is not an overnight-ready GPU result.


## Start the overnight work
This runs Parts 1 and 3 sequentially, never concurrently. Successful unchanged items are reused after an interruption. Runtime failures remain visible, not converted into successful results. The executed notebook files and exports are on the runtime filesystem; a local/Docker runtime does not automatically save files into browser Downloads.

In [3]:
try:
    run_script("tools/overnight_runner.py", *(["--smoke-only"] if SMOKE_ONLY else []))
finally:
    os.environ.pop("HF_TOKEN", None)
print("Executed notebooks:", ROOT / "output/executed")
print("Result ZIPs:", ROOT / "output/exports")
print("Execution progress:", ROOT / "runtime_checks/overnight_progress.json")

Running monitored_run.py


Part1_RAG_Run.ipynb cell 2 of 14


{


  "commit": "a0d7dda",


  "working_tree_dirty": true,


  "python": "3.12.6",


... (61 outputs left out)

Measured stage record: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/runtime_checks/overnight_runner_1793993175471060000.memory.json


[IPKernelApp] WARNING | Parent appears to have exited, shutting down.


[IPKernelApp] WARNING | Parent appears to have exited, shutting down.


Executed notebooks: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/output/executed
Result ZIPs: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/output/exports
Execution progress: /Users/denishatank/Documents/ChatGPT/Gen AI/Lab-2/runtime_checks/overnight_progress.json


## After the run
Keep the completed notebooks and ZIPs. Inspect the saved failures and fill the real-rating forms. Those review steps do not require repeating training or completed generation. The lab is not submission-ready merely because the computational stages finished.